# 05.3 Complex Numbers

Python is one of very few mainstream languages with **complex numbers built into
the core** — no import, no library. If you never touch signal processing,
electrical engineering or certain kinds of geometry, you may never need them.

They are worth ten minutes anyway, because they explain some surprising results
elsewhere (like `(-8) ** (1/3)`).

## Theory

### What a complex number is

A complex number has a **real** part and an **imaginary** part:

```
z = 3 + 4j
```

The `j` suffix marks the imaginary part. Mathematicians write `i`; engineers
write `j`, and Python followed the engineers because `i` is so commonly a loop
variable.

The defining property is:

```
j × j = -1
```

That single rule is what makes square roots of negative numbers possible.

### Why they exist in the core

Python's numeric tower (from PEP 3141) is:

```
Number -> Complex -> Real -> Rational -> Integral
```

`complex` sits at the top, so `int` and `float` are both *kinds of* complex
number with a zero imaginary part. Having it built in makes that hierarchy
consistent.

### Where they are used

- Signal processing and Fourier transforms
- Electrical engineering (AC circuit analysis)
- Control systems and stability analysis
- Fractals (the Mandelbrot set is defined on the complex plane)
- Quantum mechanics

In [ ]:
# Two ways to write a complex number.
from_literal = 3 + 4j
from_constructor = complex(3, 4)

print("3 + 4j          ->", from_literal)
print("complex(3, 4)   ->", from_constructor)
print("equal?", from_literal == from_constructor)
print("type:", type(from_literal).__name__)

# The parts are always floats, even when written as ints.
print("")
print("   .real      ->", from_literal.real, type(from_literal.real).__name__)
print("   .imag      ->", from_literal.imag, type(from_literal.imag).__name__)
print("   .conjugate ->", from_literal.conjugate())

# A bare j needs a number in front of it.
print("")
print("1j  is valid ->", 1j)
print("A bare `j` would be a NameError - it needs a coefficient.")

In [ ]:
# Arithmetic follows the normal algebraic rules, with j*j = -1.
first = 3 + 4j
second = 1 - 2j

print("first  =", first)
print("second =", second)
print("")
print("   addition:       ", first + second)
print("   subtraction:    ", first - second)
print("   multiplication: ", first * second)
print("   division:       ", first / second)

# Verify the defining property.
print("")
print("The defining rule:")
print("   1j * 1j =", 1j * 1j)

# Square roots of negatives now work.
print("")
print("Square root of a negative number:")
print("   (-1) ** 0.5   ->", (-1) ** 0.5)
print("   (-4) ** 0.5   ->", (-4) ** 0.5)

import math
try:
    math.sqrt(-1)
except ValueError as error:
    print("   math.sqrt(-1) ->", error, "<- math module refuses")

import cmath
print("   cmath.sqrt(-1)->", cmath.sqrt(-1), "<- cmath handles complex")

### The cube-root surprise

This is where complex numbers explain a result you might otherwise find baffling.

In [ ]:
# Asking for the cube root of -8 does not give -2.
result = (-8) ** (1 / 3)

print("(-8) ** (1/3) =", result)
print("Expected -2, got a complex number.")

print("")
print("WHY: there are THREE cube roots of -8 in the complex plane, and")
print("Python returns the 'principal' one, which happens to be complex.")

# All three roots.
import cmath
print("")
print("All three cube roots of -8:")
for index in range(3):
    angle = (cmath.pi + 2 * cmath.pi * index) / 3
    root = 2 * cmath.exp(1j * angle)
    print(f"   root {index + 1}: {root:.4f}")

print("")
print("To get the real cube root, handle the sign yourself:")

def real_cube_root(value):
    """Return the real cube root, including for negative values."""
    # Take the root of the magnitude, then restore the sign.
    if value < 0:
        return -((-value) ** (1 / 3))
    return value ** (1 / 3)


print("   real_cube_root(-8) ->", real_cube_root(-8))
print("   real_cube_root(27) ->", real_cube_root(27))

## The `cmath` module

`math` functions reject complex input. `cmath` provides complex equivalents.

In [ ]:
import cmath

value = 3 + 4j

print("Polar form - magnitude and angle:")
print("   abs(3+4j)        ->", abs(value), "<- magnitude (hypotenuse)")
print("   cmath.phase(...)  ->", cmath.phase(value), "radians")
print("   cmath.polar(...)  ->", cmath.polar(value))

# Convert back from polar.
magnitude, angle = cmath.polar(value)
print("   cmath.rect(r, a)  ->", cmath.rect(magnitude, angle))

print("")
print("Common cmath functions:")
print("   cmath.sqrt(-1)   ->", cmath.sqrt(-1))
print("   cmath.exp(1j*pi) ->", cmath.exp(1j * cmath.pi))
print("   cmath.log(-1)    ->", cmath.log(-1))

# Euler's identity, the most famous equation in mathematics.
euler = cmath.exp(1j * cmath.pi) + 1
print("")
print("Euler's identity: e^(i*pi) + 1 =", euler)
print("   isclose to zero?", cmath.isclose(euler, 0, abs_tol=1e-15))
print("   (not exactly zero because of float precision - see 05.2)")

## A practical example: the Mandelbrot set

The Mandelbrot set is defined entirely with complex arithmetic, which makes it a
compact demonstration.

In [ ]:
def in_mandelbrot_set(candidate, max_iterations=50):
    """Test whether a complex point stays bounded under z = z^2 + c."""
    # Start at zero and iterate.
    current = 0

    for iteration in range(max_iterations):
        current = current * current + candidate

        # If the magnitude exceeds 2, it will escape to infinity.
        if abs(current) > 2:
            return False, iteration

    return True, max_iterations


# Test a few known points.
test_points = [
    (0 + 0j, "origin - inside"),
    (-1 + 0j, "inside"),
    (0.5 + 0.5j, "outside"),
    (2 + 2j, "far outside"),
]

print("Testing points against the Mandelbrot set:")
print("")
for point, description in test_points:
    inside, iterations = in_mandelbrot_set(point)
    verdict = "inside" if inside else f"escaped after {iterations}"
    print(f"   {str(point):<12} {verdict:<22} ({description})")

# Render a small ASCII view.
print("")
print("A tiny ASCII rendering:")
for imaginary in [1.0, 0.5, 0.0, -0.5, -1.0]:
    row = ""
    for real in [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5]:
        inside, _ = in_mandelbrot_set(complex(real, imaginary), 30)
        row += "#" if inside else "."
    print("   " + row)

## Takeaways

1. Complex numbers are **built into Python** — no import needed for the type
   itself.
2. Write them with a `j` suffix: `3 + 4j`. Engineers use `j`; `i` is too common
   as a loop variable.
3. `.real`, `.imag` and `.conjugate()` are always **floats**.
4. The defining rule `1j * 1j == -1` is what makes roots of negatives possible.
5. `math` rejects complex values; **`cmath`** provides complex versions.
6. `(-8) ** (1/3)` returns a **complex** principal root, not `-2`.
7. `abs(z)` gives the magnitude; `cmath.polar()` gives magnitude and angle.

## Try it yourself

1. Verify `(3 + 4j) * (3 - 4j)` equals `25`. Why is the result real?
2. Compute `abs(3 + 4j)`. Which famous triangle is this?
3. Try `math.sqrt(-1)` then `cmath.sqrt(-1)`. Why do they differ?
4. Extend the Mandelbrot renderer to a finer grid.